## Setup: Imports and Drive Connection

In [ ]:
# here we are importing all the required libraries
import tensorflow as tf
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout, RandomFlip, RandomRotation
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
import matplotlib.pyplot as plt
import os
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# here we are connecting to google drive
print("connecting to google drive...")
from google.colab import drive
drive.mount('/content/drive')
print("drive mounted successfully")


Connecting to Google Drive...
Mounted at /content/drive
Drive mounted successfully.


##Load Data: Unzip and Define Paths
##Next, we'll unzip our DATA.zip file.

In [ ]:
# here we are unzipping the dataset from google drive
print("unzipping the data zip file from drive")
ZIP_PATH = "/content/drive/MyDrive/DATA.zip"
!unzip -o -q {ZIP_PATH} -d "/content/"
print("data is unzipped and ready in content")

# here we are defining the main project variables
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS_STAGE_1 = 15
EPOCHS_STAGE_2 = 10

# here we are defining the paths to our training validation and testing folders
TRAIN_DIR = "/content/DATA/Training(70%)"
VALID_DIR = "/content/DATA/Validation(20%)"
TEST_DIR = "/content/DATA/Testing(10%)"


Unzipping the DATA.zip file from Drive...
Data is unzipped and ready in /content/.


## Create Data Pipelines (Initial Load)

In [ ]:
# here we are loading the training validation and test datasets
print("loading training validation and test datasets")

# here we are loading the training dataset to check its labels
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, label_mode="categorical", seed=123,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)

# here we are loading the validation dataset
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR, label_mode="categorical", seed=123,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)

# here we are loading the test dataset without shuffling
test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, label_mode="categorical", image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE, shuffle=False
)

# here we are printing the class names to verify
class_names = train_dataset.class_names
print(f"our class names are: {class_names}")


Loading Training, Validation, and Test datasets...
Found 2297 files belonging to 4 classes.
Found 573 files belonging to 4 classes.
Found 394 files belonging to 4 classes.
Our class names are: ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']


## Calculate Class Weights
This is our key experiment. We saw our no_tumor class has half as many images as the others. This imbalance is making our model biased.


In [ ]:
# here we are calculating class weights to handle data imbalance
print("calculating class weights to fix data imbalance")

# here we are getting all the labels from the training dataset
train_labels = []
for images, labels in train_dataset.unbatch():
    train_labels.append(np.argmax(labels.numpy()))

# here we are using sklearn to calculate balanced class weights
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

# here we are converting the weights into a dictionary for tensorflow
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

# here we are printing the calculated class weights
print(f"calculated weights: {class_weight_dict}")


Calculating class weights to fix data imbalance...
Calculated weights: {0: np.float64(0.8687594553706506), 1: np.float64(0.8727203647416414), 2: np.float64(1.817246835443038), 3: np.float64(0.8674471299093656)}


## Optimize Data Pipelines

In [ ]:
# here we are optimizing the data pipelines for faster loading
print("optimizing data pipelines")
AUTOTUNE = tf.data.AUTOTUNE

# here we are caching and prefetching the datasets to improve performance
train_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)


Optimizing data pipelines...


## Stage 1: Initial Training (With Weights)


In [ ]:
# here we are building the resnet50 model
print("building the resnet50 model")
base_model = ResNet50(weights='imagenet', include_top=False,
                      input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))

# here we are starting stage 1 for feature extraction
print("freezing the resnet50 base layers for stage 1")
base_model.trainable = False

# here we are defining the model input and adding data augmentation
inputs = Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
x = RandomFlip('horizontal')(inputs)
x = RandomRotation(0.1)(x)

# here we are passing the data through the base model
x = base_model(x, training=False)

# here we are adding pooling dropout and dense layers for classification
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

# here we are creating the final model
model = Model(inputs, outputs)

# here we are compiling the model for stage 1
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# here we are starting the training for stage 1 with class weights
print("starting model training stage 1 head only with weights")
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS_STAGE_1,
    class_weight=class_weight_dict,
    verbose=1
)

# here we are finishing the training for stage 1
print("stage 1 training complete")


Building the ResNet50 model...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Freezing the ResNet50 base layers for Stage 1.

--- Starting Model Training (Stage 1: Head Only, WITH WEIGHTS) ---
Epoch 1/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 29s 220ms/step - accuracy: 0.4556 - loss: 1.2551 - val_accuracy: 0.8255 - val_loss: 0.5270
Epoch 2/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 11s 153ms/step - accuracy: 0.7666 - loss: 0.6329 - val_accuracy: 0.8342 - val_loss: 0.4732
Epoch 3/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 11s 157ms/step - accuracy: 0.7841 - loss: 0.5206 - val_accuracy: 0.8778 - val_loss: 0.3850
Epoch 4/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 11s 156ms/step - accuracy: 0.8127 - loss: 0.4473 - val_accuracy: 0.8761 - val_loss: 0.3629
Epoch 5/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 11s 158ms/step - accuracy: 0.8199 - loss: 0.4288 - val_accuracy: 0.8901 - val_loss: 0.3435
Epoch 6/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 12s 162ms/step - accuracy: 0.8295 - loss: 0.4043 - val_accuracy: 0.8935 - val_loss: 0.3280
Epoch 7/15
72/72 ━━━━━━━━━━━━

## Stage 2: Fine-Tuning (With Weights)


In [ ]:
# here we are starting stage 2 fine tuning with class weights
print("starting model training stage 2 fine tuning with weights")
print("unfreezing the top 30 layers of the model...")
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# here we are recompiling the model for fine tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

# here we are confirming the model is ready for fine tuning
print("model recompiled for fine tuning")
model.summary()

# here we are continuing the training to fine tune the unfrozen layers
print("continuing training to fine tune the unfrozen layers...")
history_finetune = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS_STAGE_2,
    initial_epoch=history.epoch[-1],
    class_weight=class_weight_dict,
    verbose=1
)

# here we are finishing the fine tuning stage
print("stage 2 fine tuning complete")



--- Starting Model Training (Stage 2: Fine-Tuning, WITH WEIGHTS) ---
Unfreezing the top 30 layers of the model...
--- Model Re-compiled for Fine-Tuning ---


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 224, 224, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         8,196 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,595,908 (90.01 MB)

 Trainable params: 14,458,372 (55.15 MB)

 Non-trainable params: 9,137,536 (34.86 MB)

Continuing training to fine-tune the unfrozen layers...
--- Stage 2 Fine-Tuning Complete ---


## Final Evaluation and Saving


In [ ]:
# here we are evaluating the weighted fine tuned model
print("evaluating the weighted fine tuned model on the test set")
results = model.evaluate(test_dataset, verbose=1)

# here we are calculating all the evaluation metrics
metrics = {}
metrics['loss'] = results[0]
metrics['accuracy'] = results[1]
metrics['precision'] = results[2]
metrics['recall'] = results[3]

# here we are calculating the f1 score
if (metrics['precision'] + metrics['recall']) > 0:
    metrics['f1_score'] = 2 * (metrics['precision'] * metrics['recall']) / (metrics['precision'] + metrics['recall'])
else:
    metrics['f1_score'] = 0.0

# here we are printing the test results
print("weighted fine tuned model test results")
print(f"test loss: {metrics['loss']:.4f}")
print(f"test accuracy: {metrics['accuracy']:.4f}")
print(f"test precision: {metrics['precision']:.4f}")
print(f"test recall: {metrics['recall']:.4f}")
print(f"test f1 score: {metrics['f1_score']:.4f}")

# here we are saving the final model to google drive
print("saving the model to google drive")
os.makedirs("/content/drive/MyDrive/MODELS", exist_ok=True)
model.save("/content/drive/MyDrive/MODELS/resnet_finetuned_weighted.h5")

# here we are confirming that the model has been saved successfully
print("weighted fine tuned resnet model saved")
print("this result should be much better")



--- Evaluating the WEIGHTED Fine-Tuned Model on the Test Set ---
13/13 ━━━━━━━━━━━━━━━━━━━━ 7s 158ms/step - accuracy: 0.5058 - loss: 2.1553 - precision: 0.5145 - recall: 0.4910



--- WEIGHTED Fine-Tuned Model Test Results ---
Test Loss: 1.2766
Test Accuracy: 0.6777
Test Precision: 0.6834
Test Recall: 0.6574
Test F1-Score: 0.6701

--- Saving the model to Google Drive ---
WEIGHTED Fine-Tuned ResNet model saved.
This result should be much better!
